# Capacitated Vehicle Routing Problem (CVRP)

Two identical trucks start at the **NYC** depot. Each customer needs exactly one pickup, every truck returns to NYC, and no truck may carry more than its capacity. Choose the routes that minimize the total distance traveled by both trucks.

In [16]:
# Define packages
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

In [17]:
# DISTANCE data
NYC, PHL, DC, BOS = 'NYC', 'PHL', 'DC', 'BOS'
DEPOT = NYC

distance_data = pd.DataFrame(columns=['From', 'To', 'Distance'])
distance_data['From']     = [NYC, NYC, NYC, NYC, PHL, PHL, PHL, PHL, DC, DC, DC, DC, BOS, BOS, BOS, BOS]
distance_data['To']       = [NYC, PHL, DC, BOS, NYC, PHL, DC, BOS, NYC, PHL, DC, BOS, NYC, PHL, DC, BOS]
distance_data['Distance'] = [0, 97, 229, 215, 97, 0, 142, 308, 229, 142, 0, 443, 215, 308, 443, 0]
distance_data

,From,To,Distance
0,NYC,NYC,0
1,NYC,PHL,97
2,NYC,DC,229
3,NYC,BOS,215
4,PHL,NYC,97
5,PHL,PHL,0
6,PHL,DC,142
7,PHL,BOS,308
8,DC,NYC,229
9,DC,PHL,142


In [18]:
# PICKUP and FLEET data
demand_data = pd.DataFrame(columns=['Customer', 'Pickup'])
demand_data['Customer'] = [BOS, PHL, DC]
demand_data['Pickup']   = [120, 80, 70]

TRUCKS   = 2
CAPACITY = 150
demand_data

,Customer,Pickup
0,BOS,120
1,PHL,80
2,DC,70


In [4]:
# Create empty model
m = AbstractModel()

**Sets and parameters**

- $V$: set of locations (depot and customers)
- $C = V \setminus \{NYC\}$: set of customers
- $A = \{(i,j) \in V \times V : i \ne j\}$: set of arcs (no self-loops)
- $c_{ij}$: distance from location $i$ to location $j$
- $d_i$: quantity picked up at customer $i$
- $K$: number of trucks
- $Q$: truck capacity

In [5]:
# Define Sets
m.V = Set()
m.C = Set(within=m.V, initialize=lambda m: [i for i in m.V if i != DEPOT])
m.A = Set(within=m.V * m.V, initialize=lambda m: [(i, j) for i in m.V for j in m.V if i != j])

# Define Params
m.c = Param(m.V, m.V)
m.d = Param(m.C)
m.K = Param()
m.Q = Param()

**Decision variables**

- $x_{ij} \in \{0,1\}$: 1 if a truck travels directly from $i$ to $j$. The trucks are identical, so no truck index is needed: each selected departure from NYC starts one truck's route.
- $u_i$: cumulative load on the truck after its pickup at customer $i$ (used by the load-propagation constraints)

$d_i \le u_i \le Q \qquad \forall i \in C$

In [6]:
# Define Decision Vars
m.x = Var(m.A, domain=Binary)
m.u = Var(m.C, domain=NonNegativeReals, bounds=lambda m, i: (m.d[i], m.Q))

**Enter each customer exactly once**

$\sum_{i \in V,\, i \ne j} x_{ij} = 1 \qquad \forall j \in C$

In [7]:
def Enter_Once_Rule(m, j):
    return sum(m.x[i, j] for i in m.V if i != j) == 1
m.EnterOnce = Constraint(m.C, rule=Enter_Once_Rule)

**Leave each customer exactly once**

$\sum_{j \in V,\, j \ne i} x_{ij} = 1 \qquad \forall i \in C$

In [8]:
def Leave_Once_Rule(m, i):
    return sum(m.x[i, j] for j in m.V if j != i) == 1
m.LeaveOnce = Constraint(m.C, rule=Leave_Once_Rule)

**Departures from the depot**

One route leaves NYC for each truck.

$\sum_{j \in C} x_{NYC,\,j} = K$

In [ ]:
def Depot_Departures_Rule(m):
    return sum(m.x[DEPOT, j] for j in m.C) == m.K
m.DepotDepartures = Constraint(rule=Depot_Departures_Rule)

**Returns to the depot**

Every route ends at NYC.

$\sum_{i \in C} x_{i,\,NYC} = K$

In [10]:
def Depot_Returns_Rule(m):
    return sum(m.x[i, DEPOT] for i in m.C) == m.K
m.DepotReturns = Constraint(rule=Depot_Returns_Rule)

**Load propagation (capacity and subtour elimination)**

If a truck goes from customer $i$ to customer $j$, then the load after $j$ must include $j$'s pickup ($u_j \ge u_i + d_j$). Otherwise the constraint is not binding. Together with the load bounds, this enforces capacity and eliminates customer-only cycles.

$u_i - u_j + Q\, x_{ij} \le Q - d_j \qquad \forall (i,j) \in A,\; i, j \ne NYC$

In [11]:
def Load_Rule(m, i, j):
    if i == DEPOT or j == DEPOT:
        return Constraint.Skip
    return m.u[i] - m.u[j] + m.Q * m.x[i, j] <= m.Q - m.d[j]
m.Load = Constraint(m.A, rule=Load_Rule)

**Minimize total distance**

$\min \sum_{(i,j) \in A} c_{ij}\, x_{ij}$

In [12]:
def Min_Distance_Rule(m):
    return sum(m.c[i, j] * m.x[i, j] for (i, j) in m.A)
m.Min_Distance = Objective(rule=Min_Distance_Rule, sense=minimize)

In [13]:
# Build and read instance data
instanceData = {None: {
    'V': {None: list(distance_data['From'].unique())},
    'c': distance_data.set_index(['From', 'To'])['Distance'].to_dict(),
    'd': demand_data.set_index('Customer')['Pickup'].to_dict(),
    'K': {None: TRUCKS},
    'Q': {None: CAPACITY},
}}
instance = m.create_instance(instanceData)

In [19]:
# Solve instance
solver = SolverFactory('glpk')
solution = solver.solve(instance)
print('Solver Status:')
print(solution['Solver'])
print('Optimum Objective Function Value:')
print(value(instance.Min_Distance))

Solver Status:

- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 3
      Number of created subproblems: 3
  Error rc: 0
  Time: 0.0879359245300293

Optimum Objective Function Value:
898.0


In [21]:
# Create output data: start one route at each selected departure from the depot and follow the arcs back to NYC
next_city = {i: j for (i, j) in instance.A if i != DEPOT and value(instance.x[i, j]) > 0.5}
first_stops = [j for j in instance.C if value(instance.x[DEPOT, j]) > 0.5]

rows = []
print('===== OPTIMAL ROUTES =====')
for truck, first in enumerate(first_stops, start=1):
    route = [DEPOT, first]
    while route[-1] != DEPOT:
        route.append(next_city[route[-1]])

    load = 0
    for k, (i, j) in enumerate(zip(route[:-1], route[1:])):
        load += instance.d[j] if j != DEPOT else 0
        rows.append({'Truck': truck, 'Stop': k + 1, 'From': i, 'To': j,
                     'Distance': instance.c[i, j], 'Load': load})

    route_distance = sum(instance.c[i, j] for i, j in zip(route[:-1], route[1:]))
    print(f'Truck {truck}: ' + ' → '.join(route) + f'   Load: {load}   Distance: {route_distance}')

print(f'Total Distance: {value(instance.Min_Distance):.0f}')
output_routes = pd.DataFrame(rows)
output_routes

===== OPTIMAL ROUTES =====
Truck 1: NYC → DC → PHL → NYC   Load: 150   Distance: 468
Truck 2: NYC → BOS → NYC   Load: 120   Distance: 430
Total Distance: 898


,Truck,Stop,From,To,Distance,Load
0,1,1,NYC,DC,229,70
1,1,2,DC,PHL,142,150
2,1,3,PHL,NYC,97,150
3,2,1,NYC,BOS,215,120
4,2,2,BOS,NYC,215,120
